# Match COFA → REFES — Notebook completa

Cruza el padrón COFA con los establecimientos REFES para identificar a qué registro REFES corresponde cada farmacia COFA.

### Dos caminos
- **Camino A:** COFA con CUIT que existe en REFES → agrupa por CUIT, elige la mejor REFES por scoring
- **Camino B:** COFA sin CUIT en REFES → agrupa por departamento INDEC, elige la mejor REFES sin CUIT por scoring

### Cambios respecto a versiones anteriores
- Provincia actúa como filtro duro: si no matchea, score_total = 0
- Ponderaciones revisadas: domicilio y nombre suben, localidad baja
- Lógica especial para CABA: localidad ignorada, domicilio y coordenadas dominan
- Score de domicilio con umbral mínimo: si < 30 puntos, se descarta (ruido)
- Coordenadas geográficas: si REFES tiene lat/lon, se calcula distancia haversine como score adicional
- Un REFES por establecimiento garantizado: el greedy excluye REFES ya asignadas en primera ronda
- Pregunta explícita sobre disponibilidad de coordenadas en el archivo externo

### Inputs
- `padron_completo_farmacias_2026.xlsx`
- `departamento_provincia_cofa_COMPLETO_CON_OBSERVACIONES.xlsx`
- `REFES_farmacias_cuit.csv`

### Output
- `COFA_match_REFES_completo.xlsx`

In [ ]:
import pandas as pd
import re
import math
import unicodedata
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

## 1. Parámetros

In [ ]:
# --- Rutas ---
PATH_PADRON  = 'padron_completo_farmacias_2026.xlsx'
PATH_INDEC   = 'departamento_provincia_cofa_COMPLETO_CON_OBSERVACIONES.xlsx'
PATH_REFES   = 'REFES_farmacias_cuit.csv'
PATH_OUTPUT  = 'COFA_match_REFES_completo.xlsx'

# --- ¿El archivo externo (COFA) tiene coordenadas geográficas? ---
COFA_TIENE_COORDENADAS = False
COL_LAT_COFA = None
COL_LON_COFA = None

# --- Ponderaciones Camino A (ancla CUIT) ---
# Provincia: FILTRO DURO — si no matchea, score_total = 0.
# Domicilio: score mejorado con bonus por número de puerta exacto.
WA_DEPT  = 20   # departamento
WA_LOC   =  5   # localidad (poco discriminante, puede variar mucho)
WA_NOM   = 35   # nombre del establecimiento
WA_DOM   = 40   # domicilio — cualquier coincidencia suma, número de puerta pesa más
assert WA_DEPT + WA_LOC + WA_NOM + WA_DOM == 100

# --- Ponderaciones Camino B (sin ancla CUIT, más peso en geo) ---
WB_DEPT  = 25
WB_LOC   =  5
WB_NOM   = 30
WB_DOM   = 40
assert WB_DEPT + WB_LOC + WB_NOM + WB_DOM == 100

# --- Ponderaciones CABA (provincia_id == '02') ---
# Localidad no discrimina en CABA. Domicilio y coordenadas dominan.
WCABA_NOM   = 30
WCABA_DOM   = 45
WCABA_COORD = 25
assert WCABA_NOM + WCABA_DOM + WCABA_COORD == 100

# --- Bonus por geo exacta ---
# Cuando provincia Y departamento coinciden por código INDEC exacto,
# se suma este bonus al score total. Refleja que la geo ya está garantizada
# y cualquier señal adicional en nombre/domicilio es suficiente.
# Ejemplo: ITALIANA (prov+dept código) base=66 + 15 = 81 → BUENO
# Ejemplo: SAN MARTIN 6 (mismo CUIT, geo ok) base=42.5 + 15 = 57.5 → REVISAR ✓
BONUS_GEO_EXACTA = 15   # puntos extra cuando prov+dept coinciden por código INDEC
BONUS_DOM_EXACTO = 10   # puntos extra cuando domicilio coincide al 100% — dirección exacta es señal muy fuerte

# --- Bonus de domicilio por número de puerta ---
# Si el número de puerta coincide exactamente pero el score base es bajo
# (por typo en el nombre de la calle), se suma este bonus al score de domicilio.
# Ejemplo: "GOYCOECHEA 615" vs "GOYCOCHEA 615" → base=33, bonus=30 → final=63
BONUS_NUM_PUERTA = 30    # puntos extra cuando número coincide y base < umbral
UMBRAL_BONUS_DOM = 50    # solo aplica el bonus si el score base es menor a este valor

# --- Umbrales de calidad final ---
UMBRAL_BUENO   = 70
UMBRAL_REVISAR = 45   # bajado de 50: prov+dept+algo de nom/dom ya es señal

# --- Umbral mínimo de asignación ---
# Si el mejor score disponible para una COFA es menor a este valor,
# NO se asigna ninguna REFES — queda como SIN_MATCH.
# Aplica tanto a la primera ronda (exclusiva) como a la segunda (compartida).
# Evita asignaciones claramente incorrectas como MITRE 4 → OZAMIS 401 (score ~8).
UMBRAL_MINIMO_ASIGNACION = 30   # score mínimo para asignar — por debajo es ruido


## 2. Funciones de normalización y scoring

In [ ]:
def clean_cuit(s):
    """Extrae dígitos del CUIT. Devuelve None si tiene menos de 10 dígitos."""
    if pd.isna(s):
        return None
    c = re.sub(r'[^0-9]', '', str(s))
    return c if len(c) >= 10 else None


def to_indec(x, n):
    """Convierte un código INDEC a string zero-padded de n dígitos."""
    try:
        return str(int(float(x))).zfill(n)
    except (ValueError, TypeError):
        return None


def norm(s):
    """Normalización general: mayúsculas, sin tildes, sin puntuación, sin espacios dobles."""
    if pd.isna(s) or str(s).strip() in ('nan', 'None', ''):
        return ''
    s = str(s).upper().strip()
    s = unicodedata.normalize('NFD', s)
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    s = re.sub(r'[^\w\s]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()


def strip_prefix(s, words):
    """Elimina prefijos comunes al inicio (FARMACIA, FCIA, AV, CALLE, etc.).
    También maneja punto después del prefijo: 'FCIA. DEL CENTRO' → 'DEL CENTRO'.
    """
    for w in words:
        s = re.sub(r'^' + w + r'[\s\.]+', '', s)
    return s.strip()


def token_set_ratio(a, b):
    """
    Similitud por intersección/unión de tokens.
    Ignora orden, tolera palabras extra. Devuelve 0–100.
    """
    if not a or not b:
        return 0.0
    sa, sb = set(a.split()), set(b.split())
    inter = len(sa & sb)
    union = len(sa | sb)
    return round(inter / union * 100, 1) if union else 0.0


# Mapeo de números romanos a arábigos (hasta 20, suficiente para nombres de farmacias)
ROMAN_MAP = {
    'XVIII':'18','XVII':'17','XVI':'16','XV':'15','XIV':'14','XIII':'13',
    'XII':'12','XI':'11','X':'10','IX':'9','VIII':'8','VII':'7','VI':'6',
    'V':'5','IV':'4','III':'3','II':'2','I':'1',
}

def replace_roman(s):
    """
    Convierte números romanos a arábigos cuando aparecen como token completo.
    Ejemplo: 'SAN MARTIN II' → 'SAN MARTIN 2'
    Evita reemplazos parciales: 'VILLA' no se convierte por la I al final.
    """
    for rom, arab in ROMAN_MAP.items():
        s = re.sub(r'\b' + rom + r'\b', arab, s)
    return s


def exact_code(a, b):
    """100 si los códigos son iguales, 0 si no."""
    if not a or not b:
        return 0.0
    return 100.0 if str(a) == str(b) else 0.0


def extract_numbers(s):
    """Extrae todos los números de una cadena (para comparar número de puerta)."""
    return set(re.findall(r'\b\d+\b', s or ''))


def dom_score(dom_a, dom_b):
    """
    Score de similitud de domicilio mejorado.

    Combina token_set_ratio con un bonus por coincidencia exacta de número de puerta.

    El problema que resuelve: "GOYCOECHEA 615" vs "GOYCOCHEA 615" (typo en la calle)
    tiene un token_set_ratio base bajo (~33) porque los tokens no coinciden exactamente.
    Pero el número 615 sí coincide → señal fuerte de que es la misma dirección.

    Si número coincide Y base < UMBRAL_BONUS_DOM → se suma BONUS_NUM_PUERTA.
    Ejemplo: base=33 + bonus=30 → score_dom=63 (correcto: es la misma puerta)

    Si el número NO coincide (1164 vs 1180), no se aplica el bonus → el score
    refleja honestamente la incertidumbre (puede ser diferente local).
    """
    if not dom_a or not dom_b:
        return 0.0

    sa, sb = set(dom_a.split()), set(dom_b.split())
    inter = len(sa & sb)
    union = len(sa | sb)
    base = inter / union * 100 if union else 0.0

    nums_a = extract_numbers(dom_a)
    nums_b = extract_numbers(dom_b)
    num_coincide = bool(nums_a & nums_b) if nums_a and nums_b else False

    bonus = BONUS_NUM_PUERTA if (num_coincide and base < UMBRAL_BONUS_DOM) else 0
    return round(min(100.0, base + bonus), 1)


def haversine_km(lat1, lon1, lat2, lon2):
    """Distancia en km entre dos puntos geográficos (fórmula haversine)."""
    try:
        R = 6371
        phi1, phi2 = math.radians(float(lat1)), math.radians(float(lat2))
        dphi = math.radians(float(lat2) - float(lat1))
        dlam = math.radians(float(lon2) - float(lon1))
        a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
        return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    except (TypeError, ValueError):
        return None


def coord_score(km):
    """
    Convierte distancia en km a score 0-100.
    0 km → 100 | 1 km → ~86 | 3 km → ~64 | 5 km → ~47 | 10 km → ~22
    """
    if km is None:
        return None
    return round(max(0.0, 100.0 * math.exp(-0.15 * km)), 1)


def score_pair(c, r, weights_key):
    """
    Score de similitud entre una fila COFA (c dict) y una fila REFES (r dict).

    weights_key: 'A', 'B' o 'CABA'

    Reglas:
    - Provincia es filtro DURO: si no matchea (score == 0), score_total = 0.
    - Domicilio: score mejorado con bonus por número de puerta exacto.
    - CABA: localidad ignorada, ponderaciones especiales.
    - Coordenadas: si COFA_TIENE_COORDENADAS y REFES tiene lat/lon,
      se calcula score haversine y se incorpora al total.
    """
    # --- Provincia (filtro duro) ---
    if c.get('prov_indec') and r.get('prov_indec'):
        s_prov = exact_code(c['prov_indec'], r['prov_indec'])
        pm = 'codigo'
    else:
        s_prov = token_set_ratio(norm(c.get('Provincia', '')), norm(r.get('provincia_nombre', '')))
        pm = 'texto'

    # Si provincia no matchea, score = 0 — filtro duro
    if s_prov == 0.0:
        return {
            'score_provincia': 0.0, 'score_departamento': 0.0,
            'score_localidad': 0.0, 'score_nombre': 0.0,
            'score_domicilio': 0.0, 'score_coordenadas': None,
            'score_total': 0.0, 'prov_method': pm, 'dept_method': None,
        }

    # Detectar CABA (prov_indec == '02')
    es_caba = (c.get('prov_indec') == '02' or r.get('prov_indec') == '02')

    # --- Departamento ---
    if c.get('dept_indec') and r.get('dept_indec'):
        s_dept = exact_code(c['dept_indec'], r['dept_indec'])
        dm = 'codigo'
    else:
        s_dept = token_set_ratio(
            norm(c.get('Partido/ Departamento', '')),
            norm(r.get('departamento_nombre', ''))
        )
        dm = 'texto'

    # --- Localidad ---
    s_loc = 0.0 if es_caba else token_set_ratio(
        norm(c.get('Localidad', '')),
        norm(r.get('localidad_nombre', ''))
    )

    # --- Nombre (sin prefijos FARMACIA/FARM/FCIA, con conversión de números romanos) ---
    _NOM_PREFIXES = ['FARMACIA', 'FCIA', 'FARM']
    nom_c = replace_roman(strip_prefix(norm(c.get('Nombre', '')), _NOM_PREFIXES))
    nom_r = replace_roman(strip_prefix(norm(r.get('establecimiento_nombre', '')), _NOM_PREFIXES))
    s_nom = token_set_ratio(nom_c, nom_r)

    # --- Domicilio (con bonus por número de puerta) ---
    dom_c = strip_prefix(
        norm(str(c.get('Calle', '')) + ' ' + str(c.get('Nro', ''))),
        ['AVENIDA', 'AV', 'CALLE']
    )
    dom_r = strip_prefix(norm(r.get('domicilio', '')), ['AVENIDA', 'AV', 'CALLE'])
    s_dom = dom_score(dom_c, dom_r)

    # --- Coordenadas (haversine, opcional) ---
    s_coord = None
    if COFA_TIENE_COORDENADAS and COL_LAT_COFA and COL_LON_COFA:
        lat_c = c.get(COL_LAT_COFA)
        lon_c = c.get(COL_LON_COFA)
        lat_r = r.get('latitud')
        lon_r = r.get('longitud')
        if lat_c and lon_c and lat_r and lon_r:
            km = haversine_km(lat_c, lon_c, lat_r, lon_r)
            s_coord = coord_score(km)

    # --- Score total según contexto ---
    if es_caba:
        if s_coord is not None:
            total = (s_nom * WCABA_NOM + s_dom * WCABA_DOM + s_coord * WCABA_COORD) / 100
        else:
            w_nom_adj = WCABA_NOM + WCABA_COORD * 0.4
            w_dom_adj = WCABA_DOM + WCABA_COORD * 0.6
            total = (s_nom * w_nom_adj + s_dom * w_dom_adj) / 100
    else:
        if weights_key == 'A':
            W = {'DEPT': WA_DEPT, 'LOC': WA_LOC, 'NOM': WA_NOM, 'DOM': WA_DOM}
        else:
            W = {'DEPT': WB_DEPT, 'LOC': WB_LOC, 'NOM': WB_NOM, 'DOM': WB_DOM}
        total = (
            s_dept * W['DEPT'] +
            s_loc  * W['LOC']  +
            s_nom  * W['NOM']  +
            s_dom  * W['DOM']
        ) / 100

    # --- Bonus geo: prov+dept por código INDEC exacto ---
    # Cuando la geografía está garantizada por código, cualquier señal
    # en nombre/domicilio pesa más. Se suma al total, cap 100.
    geo_exacta = (pm == 'codigo' and dm == 'codigo' and s_prov == 100.0 and s_dept == 100.0)
    bonus_geo = BONUS_GEO_EXACTA if geo_exacta else 0

    # --- Bonus domicilio exacto ---
    # Cuando el domicilio coincide al 100%, la dirección es la misma — señal muy fuerte.
    # Aplica aunque la geo o el nombre sean parciales.
    bonus_dom = BONUS_DOM_EXACTO if s_dom == 100.0 else 0

    total = min(100.0, total + bonus_geo + bonus_dom)

    return {
        'score_provincia':    round(s_prov, 1),
        'score_departamento': round(s_dept, 1),
        'score_localidad':    round(s_loc,  1),
        'score_nombre':       round(s_nom,  1),
        'score_domicilio':    round(s_dom,  1),
        'score_coordenadas':  s_coord,
        'score_total':        round(total,  1),
        'geo_exacta':         geo_exacta,
        'prov_method':        pm,
        'dept_method':        dm,
    }


def greedy_match(c_list, r_list, weights_key):
    """
    Asignación greedy 1-a-1 estricta.

    Reglas:
    - Cada REFES puede asignarse a lo sumo a UNA COFA.
    - Si dos o más COFA tienen el mismo score máximo para una REFES (empate),
      ninguna la recibe — no hay evidencia suficiente para elegir.
    - COFA sin REFES exclusiva disponible → SIN_MATCH. No hay segunda ronda.

    Esto garantiza que ningún código REFES aparece dos veces en el output.
    """
    pairs = []
    for c in c_list:
        for r in r_list:
            sc = score_pair(
                c if isinstance(c, dict) else c.to_dict(),
                r if isinstance(r, dict) else r.to_dict(),
                weights_key
            )
            pairs.append((c['Idcofa'], r['establecimiento_id'], sc, c, r))

    pairs.sort(key=lambda x: -x[2]['score_total'])

    # Pre-calcular: para cada REFES, cuál es el score máximo posible
    # y cuántos COFA tienen ese score → detectar empates antes de asignar
    best_score_por_refes = {}
    for cid, rid, sc, c, r in pairs:
        st = sc['score_total']
        if st < UMBRAL_MINIMO_ASIGNACION:
            continue
        if rid not in best_score_por_refes or st > best_score_por_refes[rid]['score']:
            best_score_por_refes[rid] = {'score': st, 'count': 1}
        elif st == best_score_por_refes[rid]['score']:
            best_score_por_refes[rid]['count'] += 1

    # REFES con empate en su mejor score → bloqueadas
    refes_bloqueadas = {rid for rid, v in best_score_por_refes.items() if v['count'] > 1}

    used_c = set()
    used_r = set()
    matched = {}

    # Asignación exclusiva — una REFES, una COFA, sin segunda ronda
    for cid, rid, sc, c, r in pairs:
        if cid not in used_c and rid not in used_r:
            st = sc['score_total']
            if st < UMBRAL_MINIMO_ASIGNACION:
                continue
            if rid in refes_bloqueadas:
                continue  # empate → nadie gana esta REFES
            matched[cid] = (rid, sc, r)
            used_c.add(cid)
            used_r.add(rid)

    # COFA sin match → quedan sin asignar (SIN_MATCH en el output)
    return matched


## 3. Carga y preparación de datos

In [ ]:
padron     = pd.read_excel(PATH_PADRON, dtype=str)
cofa_indec = pd.read_excel(PATH_INDEC, dtype=str)
refes      = pd.read_csv(PATH_REFES, dtype=str)

print(f"Padrón COFA:  {len(padron):,} filas")
print(f"REFES:        {len(refes):,} filas")
print(f"INDEC mapeo:  {len(cofa_indec):,} filas")
print(f"REFES con coordenadas: {refes['longitud'].notna().sum():,} filas")

In [ ]:
# CUITs limpios
padron['cuit_limpio'] = padron['CUIT'].apply(clean_cuit)
refes['cuit_limpio']  = refes['cuit'].apply(clean_cuit)

# Códigos INDEC al padrón COFA
cofa_indec['prov_indec'] = cofa_indec['id_provincia_indec'].apply(lambda x: to_indec(x, 2))
cofa_indec['dept_indec'] = cofa_indec['id_departamento_indec'].apply(lambda x: to_indec(x, 3))

# Normalizar claves para el merge: el padrón COFA puede tener "Capital" (minúsculas)
# mientras el archivo INDEC tiene "CAPITAL" — sin esto el merge falla silenciosamente.
def norm_key(s):
    """Normaliza para usar como clave de merge: mayúsculas, sin tildes, sin espacios extra."""
    if pd.isna(s): return ''
    s = str(s).upper().strip()
    s = unicodedata.normalize('NFD', s)
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    return re.sub(r'\s+', ' ', s).strip()

padron['_key_prov']  = padron['Provincia'].apply(norm_key)
padron['_key_dept']  = padron['Partido/ Departamento'].apply(norm_key)
cofa_indec['_key_prov'] = cofa_indec['Provincia'].apply(norm_key)
cofa_indec['_key_dept'] = cofa_indec['Partido/ Departamento'].apply(norm_key)

# Deduplicar antes del merge (2 filas con Partido=NaN en Córdoba)
indec_dedup = (
    cofa_indec[['_key_prov', '_key_dept', 'prov_indec', 'dept_indec']]
    .drop_duplicates(subset=['_key_prov', '_key_dept'])
)
padron = padron.merge(indec_dedup, on=['_key_prov', '_key_dept'], how='left')
assert len(padron) == 4544, f'El merge generó filas extra: {len(padron)} != 4544'
n_con_indec = padron['prov_indec'].notna().sum()
print(f"COFA con código provincia INDEC mapeado: {n_con_indec} / {len(padron)}")

# Códigos INDEC a REFES
refes['prov_indec'] = refes['provincia_id'].apply(lambda x: to_indec(x, 2))
refes['dept_indec'] = refes['departamento_id'].apply(lambda x: to_indec(x, 3))

# Separar universos
cuits_refes  = set(refes[refes['cuit_limpio'].notna()]['cuit_limpio'])
cofa_A       = padron[padron['cuit_limpio'].isin(cuits_refes)].copy()
cofa_B       = padron[~padron['cuit_limpio'].isin(cuits_refes)].copy()
refes_con    = refes[refes['cuit_limpio'].notna()].copy()
refes_sin    = refes[refes['cuit_limpio'].isna()].copy()

print(f"COFA camino A (CUIT en REFES):    {len(cofa_A):,}")
print(f"COFA camino B (CUIT no en REFES): {len(cofa_B):,}")
print(f"REFES con CUIT: {len(refes_con):,}")
print(f"REFES sin CUIT: {len(refes_sin):,}")
print(f"\n¿COFA tiene coordenadas? {COFA_TIENE_COORDENADAS}")
if COFA_TIENE_COORDENADAS:
    print(f"  Columna lat: {COL_LAT_COFA} | Columna lon: {COL_LON_COFA}")

## 4. Camino A — matching por CUIT

In [ ]:
results_A = []

NO_SC = {
    'score_provincia': None, 'score_departamento': None,
    'score_localidad': None, 'score_nombre': None,
    'score_domicilio': None, 'score_coordenadas': None,
    'score_total': None, 'geo_exacta': None, 'prov_method': None, 'dept_method': None,
}
NO_REF = {
    'refes_id_asignado': None, 'refes_nombre_asignado': None,
    'refes_provincia': None, 'refes_prov_indec': None,
    'refes_departamento': None, 'refes_dept_indec': None,
    'refes_localidad': None, 'refes_domicilio': None,
    'refes_latitud': None, 'refes_longitud': None,
}

for cuit in cofa_A['cuit_limpio'].unique():
    c_list = [row for _, row in cofa_A[cofa_A['cuit_limpio'] == cuit].iterrows()]
    r_list = [row for _, row in refes_con[refes_con['cuit_limpio'] == cuit].iterrows()]

    matched = greedy_match(c_list, r_list, 'A')

    for c in c_list:
        cid = c['Idcofa']
        if cid in matched:
            rid, sc, r = matched[cid]
            cat = 'MATCH_BUENO' if sc['score_total'] >= UMBRAL_BUENO else 'MATCH_REVISAR'
            results_A.append({
                **c.to_dict(),
                'refes_id_asignado':     r['establecimiento_id'],
                'refes_nombre_asignado': r['establecimiento_nombre'],
                'refes_provincia':       r['provincia_nombre'],
                'refes_prov_indec':      r.get('prov_indec'),
                'refes_departamento':    r['departamento_nombre'],
                'refes_dept_indec':      r.get('dept_indec'),
                'refes_localidad':       r['localidad_nombre'],
                'refes_domicilio':       r['domicilio'],
                'refes_latitud':         r.get('latitud'),
                'refes_longitud':        r.get('longitud'),
                **sc,
                'camino': 'A',
                'categoria': cat,
            })
        else:
            # Ningún par superó UMBRAL_MINIMO_ASIGNACION → SIN_MATCH
            results_A.append({
                **c.to_dict(),
                **NO_SC, **NO_REF,
                'camino': 'A',
                'categoria': 'SIN_MATCH',
            })

df_A = pd.DataFrame(results_A)
print(f"Camino A — {len(df_A):,} COFA procesadas")
print(df_A['categoria'].value_counts())
print(f"\nScore promedio: {df_A['score_total'].mean():.1f} | mediana: {df_A['score_total'].median():.1f}")

## 5. Camino B — matching por departamento INDEC

In [ ]:
cofa_B_con = cofa_B[cofa_B['prov_indec'].notna() & cofa_B['dept_indec'].notna()].copy()
cofa_B_sin = cofa_B[cofa_B['prov_indec'].isna()  | cofa_B['dept_indec'].isna()].copy()

grupos_refes_sin = {}
for _, r in refes_sin.iterrows():
    k = (r['prov_indec'], r['dept_indec'])
    grupos_refes_sin.setdefault(k, []).append(r)

grupos_cofa_B = {}
for _, c in cofa_B_con.iterrows():
    k = (c['prov_indec'], c['dept_indec'])
    grupos_cofa_B.setdefault(k, []).append(c)

results_B = []
for grupo, c_list in grupos_cofa_B.items():
    r_list = grupos_refes_sin.get(grupo, [])
    if not r_list:
        for c in c_list:
            results_B.append({**c.to_dict(), **NO_SC, **NO_REF,
                'camino': 'B', 'categoria': 'SIN_REFES_EN_GRUPO'})
        continue

    matched = greedy_match(c_list, r_list, 'B')
    for c in c_list:
        cid = c['Idcofa']
        if cid in matched:
            rid, sc, r = matched[cid]
            cat = ('MATCH_BUENO'   if sc['score_total'] >= UMBRAL_BUENO else
                   'MATCH_REVISAR' if sc['score_total'] >= UMBRAL_REVISAR else
                   'MATCH_DUDOSO')
            results_B.append({
                **c.to_dict(),
                'refes_id_asignado':     r['establecimiento_id'],
                'refes_nombre_asignado': r['establecimiento_nombre'],
                'refes_provincia':       r['provincia_nombre'],
                'refes_prov_indec':      r.get('prov_indec'),
                'refes_departamento':    r['departamento_nombre'],
                'refes_dept_indec':      r.get('dept_indec'),
                'refes_localidad':       r['localidad_nombre'],
                'refes_domicilio':       r['domicilio'],
                'refes_latitud':         r.get('latitud'),
                'refes_longitud':        r.get('longitud'),
                **sc,
                'camino': 'B',
                'categoria': cat,
            })
        else:
            results_B.append({**c.to_dict(), **NO_SC, **NO_REF,
                'camino': 'B', 'categoria': 'SIN_MATCH'})

for _, c in cofa_B_sin.iterrows():
    results_B.append({**c.to_dict(), **NO_SC, **NO_REF,
        'camino': 'B', 'categoria': 'SIN_INDEC'})

df_B = pd.DataFrame(results_B)
print(f"Camino B — {len(df_B):,} COFA procesadas")
print(df_B['categoria'].value_counts())
matched_B = df_B[df_B['score_total'].notna()]
print(f"\nScore promedio (matcheadas): {matched_B['score_total'].mean():.1f} | mediana: {matched_B['score_total'].median():.1f}")

## 6. Consolidar ambos caminos

In [ ]:
df_total = pd.concat([df_A, df_B], ignore_index=True)

print(f"Total COFA procesadas: {len(df_total):,}")
print(f"\nDistribución general:")
print(df_total['categoria'].value_counts())

# ── Diagnóstico de calidad del padrón COFA fuente ──────────────────
# Detectar y marcar tres tipos de problemas en los datos de origen

def validar_cuit(cuit_str):
    """
    Valida el dígito verificador de un CUIT argentino (11 dígitos).
    Retorna True si es válido, False si el dígito verificador no cierra,
    None si el CUIT está vacío o tiene longitud incorrecta.
    """
    if not cuit_str:
        return None
    digits = re.sub(r'[^0-9]', '', str(cuit_str))
    if len(digits) != 11:
        return False
    factors = [5, 4, 3, 2, 7, 6, 5, 4, 3, 2]
    total = sum(int(d) * f for d, f in zip(digits[:10], factors))
    resto = total % 11
    verificador = 0 if resto == 0 else (9 if resto == 1 else 11 - resto)
    return int(digits[10]) == verificador

# 1. Sin CUIT (campo vacío o no numérico)
padron['anomalia_sin_cuit'] = padron['cuit_limpio'].isna()

# 2. CUIT con dígito verificador inválido
padron['anomalia_cuit_invalido'] = padron['cuit_limpio'].apply(
    lambda c: False if pd.isna(c) else not validar_cuit(c)
)

# 3. Idcofa nulo
padron['anomalia_idcofa_nulo'] = padron['Idcofa'].isna()

# 4. Idcofa duplicado (mismo id, distinta farmacia — error en padrón fuente)
padron['anomalia_idcofa_duplicado'] = (
    padron['Idcofa'].duplicated(keep=False) & padron['Idcofa'].notna()
)

# Columna resumen: concatena las flags activas separadas por |
def flag_anomalia(row):
    flags = []
    if row.get('anomalia_sin_cuit'):         flags.append('sin_cuit')
    if row.get('anomalia_cuit_invalido'):    flags.append('cuit_invalido')
    if row.get('anomalia_idcofa_nulo'):      flags.append('idcofa_nulo')
    if row.get('anomalia_idcofa_duplicado'): flags.append('idcofa_duplicado')
    return '|'.join(flags) if flags else ''

padron['anomalia_cofa'] = padron.apply(flag_anomalia, axis=1)

# Propagar anomalías al df_total
# Idcofa puede ser NaN → usar índice posicional como fallback
padron['_idx'] = padron.index
anomalia_map = padron.set_index('Idcofa')['anomalia_cofa'].to_dict()
df_total['anomalia_cofa'] = df_total['Idcofa'].map(anomalia_map).fillna('')

dup_idcofa = padron[padron['Idcofa'].duplicated(keep=False) & padron['Idcofa'].notna()]
dup_null   = padron[padron['Idcofa'].isna()]
dupes_output = df_total['Idcofa'].duplicated().sum()

print()
print('--- Problemas de calidad en el padrón COFA fuente ---')
print(f'  Sin CUIT:                   {padron["anomalia_sin_cuit"].sum()} filas')
print(f'  CUIT inválido (dígito):     {padron["anomalia_cuit_invalido"].sum()} filas')
print(f'  Sin Idcofa (NaN):           {padron["anomalia_idcofa_nulo"].sum()} filas')
print(f'  Idcofa duplicado:           {padron["anomalia_idcofa_duplicado"].sum()} filas ({dup_idcofa["Idcofa"].nunique()} ids)')
print(f'  Total con alguna anomalía:  {(padron["anomalia_cofa"] != "").sum()} filas')
# match_compartido ya no aplica — la asignación es siempre exclusiva
# print(f'  Matches compartidos total:  {df_total["match_compartido"].sum()}')

## 7. Exportar a Excel

In [ ]:
cofa_orig  = ['Idcofa', 'CUIT', 'cuit_limpio', 'Nombre', 'Razon Social',
               'Calle', 'Nro', 'Localidad', 'Partido/ Departamento', 'Provincia',
               'prov_indec', 'dept_indec']
score_cols = ['score_provincia', 'score_departamento', 'score_localidad',
               'score_nombre', 'score_domicilio', 'score_coordenadas', 'score_total',
               'geo_exacta', 'prov_method', 'dept_method']
refes_cols = ['refes_id_asignado', 'refes_nombre_asignado',
               'refes_provincia', 'refes_prov_indec',
               'refes_departamento', 'refes_dept_indec',
               'refes_localidad', 'refes_domicilio',
               'refes_latitud', 'refes_longitud']
meta_cols  = ['anomalia_cofa', 'camino', 'categoria']

all_cols = cofa_orig + score_cols + refes_cols + meta_cols

fill_cofa   = PatternFill('solid', start_color='1F4E79')
fill_score  = PatternFill('solid', start_color='375623')
fill_refes  = PatternFill('solid', start_color='833C00')
fill_meta   = PatternFill('solid', start_color='595959')
fill_green  = PatternFill('solid', start_color='C6EFCE')
fill_yellow = PatternFill('solid', start_color='FFEB9C')
fill_red    = PatternFill('solid', start_color='FFC7CE')
fill_gray   = PatternFill('solid', start_color='D9D9D9')

cat_fills = {
    'MATCH_BUENO':        fill_green,
    'MATCH_REVISAR':      fill_yellow,
    'MATCH_DUDOSO':       fill_red,
    'SIN_REFES_EN_GRUPO': fill_gray,
    'SIN_INDEC':          fill_gray,
}

widths = {
    'Idcofa':12,'CUIT':16,'cuit_limpio':14,'Nombre':22,'Razon Social':22,
    'Calle':20,'Nro':7,'Localidad':18,'Partido/ Departamento':20,'Provincia':14,
    'prov_indec':10,'dept_indec':10,
    'score_provincia':12,'score_departamento':14,'score_localidad':12,
    'score_nombre':12,'score_domicilio':12,'score_coordenadas':14,'score_total':11,
    'geo_exacta':10,'prov_method':10,'dept_method':10,
    'refes_id_asignado':18,'refes_nombre_asignado':25,'refes_provincia':14,
    'refes_departamento':18,'refes_localidad':18,'refes_domicilio':28,
    'refes_prov_indec':12,'refes_dept_indec':12,'refes_latitud':12,'refes_longitud':12,
    'anomalia_cofa':22,'camino':8,'categoria':20,
}


def write_data_sheet(ws, df):
    cols = [c for c in all_cols if c in df.columns]
    orden = {'MATCH_BUENO':0,'MATCH_REVISAR':1,'MATCH_DUDOSO':2,
             'SIN_REFES_EN_GRUPO':3,'SIN_INDEC':4}
    df_out = df[cols].copy()
    df_out['_ord'] = df_out['categoria'].map(orden)
    df_out = df_out.sort_values(['_ord','score_total'], ascending=[True,False]).drop(columns='_ord')
    cols = list(df_out.columns)

    ws.append(cols)
    for i, col in enumerate(cols, 1):
        cell = ws.cell(1, i)
        cell.font = Font(name='Arial', bold=True, color='FFFFFF', size=9)
        cell.alignment = Alignment(horizontal='center', wrap_text=True)
        if col in cofa_orig:    cell.fill = fill_cofa
        elif col in score_cols: cell.fill = fill_score
        elif col in refes_cols: cell.fill = fill_refes
        else:                   cell.fill = fill_meta

    cat_idx   = cols.index('categoria') + 1
    score_idx = cols.index('score_total') + 1

    for row in df_out.itertuples(index=False):
        ws.append(list(row))
        r = ws.max_row
        cat = row.categoria
        f = cat_fills.get(cat)
        if f:
            ws.cell(r, cat_idx).fill   = f
            ws.cell(r, score_idx).fill = f

    for i, col in enumerate(cols, 1):
        ws.column_dimensions[get_column_letter(i)].width = widths.get(col, 13)
    ws.freeze_panes = 'A2'
    ws.row_dimensions[1].height = 32


wb = Workbook()

# Camino A: solo registros con match (excluye SIN_MATCH — van a hoja propia)
ws_a = wb.active
ws_a.title = 'Camino A - con match'
write_data_sheet(ws_a, df_A[df_A['categoria'] != 'SIN_MATCH'])

# Camino B: solo registros con match (excluye SIN_MATCH, SIN_REFES_EN_GRUPO, SIN_INDEC)
ws_b = wb.create_sheet('Camino B - con match')
write_data_sheet(ws_b, df_B[df_B['categoria'].isin(['MATCH_BUENO','MATCH_REVISAR'])])

print("Hojas de datos escritas.")

## 8. Reportes finales

In [ ]:
def hdr(ws, row, col, text, fill_hex, font_size=10):
    cell = ws.cell(row, col, text)
    cell.font = Font(name='Arial', bold=True, color='FFFFFF', size=font_size)
    cell.fill = PatternFill('solid', start_color=fill_hex)
    cell.alignment = Alignment(horizontal='center', wrap_text=True)
    return cell

def val(ws, row, col, value, bold=False, fill=None, align='left'):
    cell = ws.cell(row, col, value)
    cell.font = Font(name='Arial', bold=bold, size=10)
    cell.alignment = Alignment(horizontal=align)
    if fill: cell.fill = fill
    return cell

F_TITLE = PatternFill('solid', start_color='1F4E79')
F_SUB   = PatternFill('solid', start_color='2E75B6')
F_GREEN = PatternFill('solid', start_color='C6EFCE')
F_YELLOW= PatternFill('solid', start_color='FFEB9C')
F_RED   = PatternFill('solid', start_color='FFC7CE')
F_GRAY  = PatternFill('solid', start_color='D9D9D9')
F_LGRAY = PatternFill('solid', start_color='F2F2F2')

# ── Hoja 3: Resumen ejecutivo ────────────────────────────────────────
ws_r = wb.create_sheet('Resumen ejecutivo')
ws_r.column_dimensions['A'].width = 42
ws_r.column_dimensions['B'].width = 16
ws_r.column_dimensions['C'].width = 16
ws_r.column_dimensions['D'].width = 16

r = 1
hdr(ws_r, r, 1, 'MATCH COFA → REFES — Resumen ejecutivo', '1F4E79', 13)
ws_r.merge_cells(f'A{r}:D{r}')
ws_r.row_dimensions[r].height = 22

r += 2
for c, txt in enumerate(['Categoría','Camino A','Camino B','Total'], 1):
    hdr(ws_r, r, c, txt, '1F4E79')

vc_A = df_A['categoria'].value_counts()
vc_B = df_B['categoria'].value_counts()

cat_order = ['MATCH_BUENO','MATCH_REVISAR','MATCH_DUDOSO','SIN_MATCH','SIN_REFES_EN_GRUPO','SIN_INDEC']
cat_labels = {
    'MATCH_BUENO':        f'Match bueno (score ≥ {UMBRAL_BUENO})',
    'MATCH_REVISAR':      f'Match revisar (score {UMBRAL_REVISAR}–{UMBRAL_BUENO-1})',
    'MATCH_DUDOSO':       f'Match dudoso (score < {UMBRAL_REVISAR})',
    'SIN_MATCH':          'Sin match disponible (REFES tomadas o empate)',
    'SIN_REFES_EN_GRUPO': 'Sin REFES en el mismo departamento (camino B)',
    'SIN_INDEC':          'Sin código INDEC mapeado (camino B)',
}
row_fills = {
    'MATCH_BUENO': F_GREEN, 'MATCH_REVISAR': F_YELLOW,
    'MATCH_DUDOSO': F_RED,  'SIN_MATCH': F_GRAY,
    'SIN_REFES_EN_GRUPO': F_GRAY, 'SIN_INDEC': F_GRAY,
}
for cat in cat_order:
    r += 1
    n_a = vc_A.get(cat, 0)
    n_b = vc_B.get(cat, 0)
    f = row_fills[cat]
    val(ws_r, r, 1, cat_labels[cat], fill=f)
    val(ws_r, r, 2, int(n_a), align='center', fill=f)
    val(ws_r, r, 3, int(n_b), align='center', fill=f)
    val(ws_r, r, 4, int(n_a+n_b), bold=True, align='center', fill=f)

r += 1
val(ws_r, r, 1, 'TOTAL', bold=True)
val(ws_r, r, 2, len(df_A), bold=True, align='center')
val(ws_r, r, 3, len(df_B), bold=True, align='center')
val(ws_r, r, 4, len(df_total), bold=True, align='center')

r += 2
hdr(ws_r, r, 1, 'Estadísticas de score (solo matcheados)', '2E75B6')
ws_r.merge_cells(f'A{r}:D{r}')
matched_A = df_A[df_A['score_total'].notna()]
matched_B = df_B[df_B['score_total'].notna()]
matched_tot = df_total[df_total['score_total'].notna()]
for label, va, vb, vt in [
    ('N matcheados', len(matched_A), len(matched_B), len(matched_tot)),
    ('Score promedio', round(matched_A['score_total'].mean(),1) if len(matched_A) else '—',
                       round(matched_B['score_total'].mean(),1) if len(matched_B) else '—',
                       round(matched_tot['score_total'].mean(),1) if len(matched_tot) else '—'),
    ('Score mediana',  round(matched_A['score_total'].median(),1) if len(matched_A) else '—',
                       round(matched_B['score_total'].median(),1) if len(matched_B) else '—',
                       round(matched_tot['score_total'].median(),1) if len(matched_tot) else '—'),
    ('Score mínimo',   round(matched_A['score_total'].min(),1) if len(matched_A) else '—',
                       round(matched_B['score_total'].min(),1) if len(matched_B) else '—',
                       round(matched_tot['score_total'].min(),1) if len(matched_tot) else '—'),
]:
    r += 1
    f = F_LGRAY if r % 2 == 0 else None
    val(ws_r, r, 1, label, fill=f)
    val(ws_r, r, 2, va, align='center', fill=f)
    val(ws_r, r, 3, vb, align='center', fill=f)
    val(ws_r, r, 4, vt, align='center', fill=f)

r += 2
hdr(ws_r, r, 1, 'Ponderaciones utilizadas', '2E75B6')
hdr(ws_r, r, 2, 'Camino A', '2E75B6')
hdr(ws_r, r, 3, 'Camino B', '2E75B6')
hdr(ws_r, r, 4, 'CABA', '2E75B6')
for label, wa, wb_, wc in [
    ('Provincia (filtro duro — si=0 → total=0)', '—', '—', '—'),
    ('Departamento', WA_DEPT, WB_DEPT, '—'),
    ('Localidad',    WA_LOC,  WB_LOC,  '0 (ignorada)'),
    ('Nombre',       WA_NOM,  WB_NOM,  WCABA_NOM),
    ('Domicilio',    WA_DOM,  WB_DOM,  WCABA_DOM),
    ('Coordenadas',  '—',     '—',     WCABA_COORD),
    ('Domicilio sin umbral mínimo (cualquier señal suma)', '—', '—', '—'),
]:
    r += 1
    f = F_LGRAY if r % 2 == 0 else None
    val(ws_r, r, 1, label, fill=f)
    val(ws_r, r, 2, f'{wa}%' if isinstance(wa, int) else str(wa), align='center', fill=f)
    val(ws_r, r, 3, f'{wb_}%' if isinstance(wb_, int) else str(wb_), align='center', fill=f)
    val(ws_r, r, 4, f'{wc}%' if isinstance(wc, int) else str(wc), align='center', fill=f)

print("Hoja 'Resumen ejecutivo' lista.")

In [ ]:
# ── Hoja 4: Scores por provincia ─────────────────────────────────────
ws_p = wb.create_sheet('Scores por provincia')
ws_p.column_dimensions['A'].width = 24
for col in 'BCDEFG': ws_p.column_dimensions[col].width = 13

r = 1
hdr(ws_p, r, 1, 'SCORES POR PROVINCIA', '1F4E79', 12)
ws_p.merge_cells(f'A{r}:G{r}')
ws_p.row_dimensions[r].height = 18

r += 2
for c, txt in enumerate(['Provincia','N matcheadas','Score prom.','Score med.',
                          f'N buenas (≥{UMBRAL_BUENO})','N revisar','N sin match'], 1):
    hdr(ws_p, r, c, txt, '2E75B6')

matched_all = df_total[df_total['score_total'].notna()].copy()
sin_match   = df_total[df_total['score_total'].isna()].copy()
provs = sorted(df_total['Provincia'].dropna().unique())

for i, prov in enumerate(provs):
    r += 1
    sub_m = matched_all[matched_all['Provincia'] == prov]
    sub_s = sin_match[sin_match['Provincia'] == prov]
    f = F_LGRAY if i % 2 == 0 else None
    val(ws_p, r, 1, prov, fill=f)
    val(ws_p, r, 2, len(sub_m), align='center', fill=f)
    val(ws_p, r, 3, round(sub_m['score_total'].mean(),1) if len(sub_m) else '—', align='center', fill=f)
    val(ws_p, r, 4, round(sub_m['score_total'].median(),1) if len(sub_m) else '—', align='center', fill=f)
    n_buenas  = int((sub_m['score_total'] >= UMBRAL_BUENO).sum())
    n_revisar = int((sub_m['score_total'] < UMBRAL_BUENO).sum())
    val(ws_p, r, 5, n_buenas,  align='center', fill=F_GREEN if n_buenas > 0 else f)
    val(ws_p, r, 6, n_revisar, align='center', fill=F_YELLOW if n_revisar > 0 else f)
    val(ws_p, r, 7, len(sub_s), align='center', fill=F_GRAY if len(sub_s) > 0 else f)

r += 1
val(ws_p, r, 1, 'TOTAL', bold=True)
val(ws_p, r, 2, len(matched_all), bold=True, align='center')
val(ws_p, r, 3, round(matched_all['score_total'].mean(),1), bold=True, align='center')
val(ws_p, r, 4, round(matched_all['score_total'].median(),1), bold=True, align='center')
val(ws_p, r, 5, int((matched_all['score_total'] >= UMBRAL_BUENO).sum()), bold=True, align='center')
val(ws_p, r, 6, int((matched_all['score_total'] <  UMBRAL_BUENO).sum()), bold=True, align='center')
val(ws_p, r, 7, len(sin_match), bold=True, align='center')

print("Hoja 'Scores por provincia' lista.")

In [ ]:
# ── Hoja 5: Sin match — todos los no asignados de ambos caminos ──────
# Incluye: SIN_MATCH (A y B), SIN_REFES_EN_GRUPO, SIN_INDEC, MATCH_DUDOSO
# Columna 'tenia_cuit' indica si el registro tenía CUIT en REFES (camino A) o no (camino B)
ws_sm = wb.create_sheet('Sin match')

cats_sin = ['SIN_MATCH', 'SIN_REFES_EN_GRUPO', 'SIN_INDEC', 'MATCH_DUDOSO']
df_sin = df_total[df_total['categoria'].isin(cats_sin)].copy()
df_sin['tenia_cuit'] = df_sin['camino'] == 'A'   # True = tenía CUIT en REFES

# Agregar columna tenia_cuit a meta_cols para el output
cols_sm_base = [c for c in all_cols if c in df_sin.columns]
# Insertar tenia_cuit antes de camino
if 'tenia_cuit' not in cols_sm_base:
    idx = cols_sm_base.index('camino') if 'camino' in cols_sm_base else len(cols_sm_base)
    cols_sm_base.insert(idx, 'tenia_cuit')

df_sin_out = df_sin[cols_sm_base].sort_values(['camino','categoria'])

ws_sm.append(list(df_sin_out.columns))
for i, col in enumerate(df_sin_out.columns, 1):
    cell = ws_sm.cell(1, i)
    cell.font = Font(name='Arial', bold=True, color='FFFFFF', size=9)
    cell.alignment = Alignment(horizontal='center', wrap_text=True)
    if col == 'tenia_cuit':      cell.fill = fill_meta
    elif col in cofa_orig:       cell.fill = fill_cofa
    elif col in score_cols:      cell.fill = fill_score
    elif col in refes_cols:      cell.fill = fill_refes
    else:                        cell.fill = fill_meta

cat_idx = list(df_sin_out.columns).index('categoria') + 1
for row in df_sin_out.itertuples(index=False):
    ws_sm.append(list(row))
    rn = ws_sm.max_row
    f = cat_fills.get(row.categoria)
    if f: ws_sm.cell(rn, cat_idx).fill = f

for i, col in enumerate(df_sin_out.columns, 1):
    ws_sm.column_dimensions[get_column_letter(i)].width = widths.get(col, 13)
ws_sm.freeze_panes = 'A2'
ws_sm.row_dimensions[1].height = 32

# ── Guardar ───────────────────────────────────────────────────────────
wb.save(PATH_OUTPUT)

# Verificar que las tres hojas de datos suman el total
n_a    = len(df_A[df_A['categoria'] != 'SIN_MATCH'])
n_b    = len(df_B[df_B['categoria'].isin(['MATCH_BUENO','MATCH_REVISAR'])])
n_sin  = len(df_sin_out)
total_check = n_a + n_b + n_sin

print(f"\nArchivo guardado: {PATH_OUTPUT}")
print(f"  Hoja 'Camino A - con match':  {n_a:,} filas")
print(f"  Hoja 'Camino B - con match':  {n_b:,} filas")
print(f"  Hoja 'Resumen ejecutivo':     OK")
print(f"  Hoja 'Scores por provincia':  {len(provs)} provincias")
print(f"  Hoja 'Sin match':             {n_sin:,} filas")
print(f"  TOTAL hojas datos:            {total_check:,}  {'✓' if total_check == len(df_total) else '✗ NO SUMA'}")
print()
print(f"Distribución 'Sin match':")
print(df_sin_out['categoria'].value_counts())
print()
print(f"Con CUIT (camino A):  {df_sin_out['tenia_cuit'].sum():,}")
print(f"Sin CUIT (camino B):  {(~df_sin_out['tenia_cuit']).sum():,}")
